# SatQuery land-cover fine-tuning (Colab)
This notebook trains a **pixel segmentation** model with geographically grouped splits. The default Sentinel-2 track uses the public DFC2020-S2 segmentation dataset and SatlasPretrain `Sentinel2_SwinB_SI_MS`. It never treats scene-classification labels as masks. Demo mode is bounded; full mode is the required final run. Existing RGB FLAIR weights in this bundle are preserved.

In [ ]:
# Install only missing packages. Colab's compatible torch is deliberately retained.
import importlib.util, subprocess, sys
required = {'torchgeo':'torchgeo==0.8.1', 'satlaspretrain_models':'satlaspretrain-models>=0.3', 'albumentations':'albumentations>=1.4,<3', 'timm':'timm>=1.0,<2'}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
import torch
print('Using Colab PyTorch', torch.__version__, '- it was not reinstalled')

In [ ]:
# Binary compatibility gate. If this fails after installation, restart Runtime and run from the top.
try:
    import numpy as np, scipy, rasterio, skimage
    from scipy import ndimage
    ndimage.label(np.ones((2,2), dtype=bool))
except (ImportError, ValueError, OSError) as exc:
    raise RuntimeError('NumPy/SciPy/rasterio binary mismatch. Choose Runtime > Restart session, then rerun all cells. Original error: ' + repr(exc))
print({'numpy':np.__version__, 'scipy':scipy.__version__, 'rasterio':rasterio.__version__, 'skimage':skimage.__version__})

In [ ]:
from pathlib import Path
MODE = 'demo'  # change to 'full' for the final training run
SAVE_TO_DRIVE = False
CFG = {
 'seed': 20260922, 'batch_size': 2 if MODE=='demo' else 8, 'epochs': 2 if MODE=='demo' else 60,
 'max_samples_per_split': 128 if MODE=='demo' else None, 'patience': 3 if MODE=='demo' else 10,
 'lr': 2e-4, 'weight_decay': 1e-4, 'num_classes': 7, 'ignore_index': 255, 'tile_size': 256}
ROOT = Path('/content/satquery_landcover')
if SAVE_TO_DRIVE:
    from google.colab import drive; drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/satquery_landcover')
ROOT.mkdir(parents=True, exist_ok=True)
CFG

## Download and geographically regroup DFC2020-S2
TorchGeo downloads the revision-pinned public archive with its published MD5 checksum. Construction is retried, caches are reused, and split grouping removes the patch suffix so neighboring patches from one scene cannot cross train/validation/test.

In [ ]:
import hashlib, re, time
from torchgeo.datasets import CopernicusBenchDFC2020S2
def open_split(split):
    for attempt in range(4):
        try: return CopernicusBenchDFC2020S2(root=str(ROOT/'data'), split=split, bands=CopernicusBenchDFC2020S2.all_bands, download=True, checksum=True)
        except Exception:
            if attempt == 3: raise
            time.sleep(2**attempt)
sources = [open_split(s) for s in ('train','val','test')]
records = [(ds, i, str(ds.files[i])) for ds in sources for i in range(len(ds))]
def geography_key(name): return re.sub(r'_p\d+.*$', '', Path(name).stem)
def assigned_split(key):
    value = int(hashlib.sha256(key.encode()).hexdigest()[:8], 16) % 100
    return 'train' if value < 70 else ('val' if value < 85 else 'test')
group_split = {geography_key(name): assigned_split(geography_key(name)) for _,_,name in records}
partitions = {split:[(ds,i,name) for ds,i,name in records if group_split[geography_key(name)]==split] for split in ('train','val','test')}
assert not (set(geography_key(x[2]) for x in partitions['train']) & set(geography_key(x[2]) for x in partitions['val']))
assert not (set(geography_key(x[2]) for x in partitions['train']) & set(geography_key(x[2]) for x in partitions['test']))
if CFG['max_samples_per_split']: partitions = {k:v[:CFG['max_samples_per_split']] for k,v in partitions.items()}
print({k:len(v) for k,v in partitions.items()}, 'geographic groups', {k:len(set(geography_key(x[2]) for x in v)) for k,v in partitions.items()})

In [ ]:
# Dataset ontology -> SatQuery canonical IDs: unknown, built-up, vegetation, cropland, bare soil, water, road/impervious.
# DFC2020 Urban/Built-up maps to built-up. It has no independent road label; class 6 requires added geographically held-out road labels before production release.
import torch, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
DFC_TO_SATQUERY = torch.tensor([255,2,2,2,2,2,3,1,255,4,5], dtype=torch.long)
class GroupedDataset(Dataset):
    def __init__(self, rows, augment=False): self.rows, self.augment = rows, augment
    def __len__(self): return len(self.rows)
    def __getitem__(self, index):
        ds, i, name = self.rows[index]; sample = ds[i]
        image = sample['image'].float(); mask = sample['mask'].long()
        mask = DFC_TO_SATQUERY[mask.clamp(0,10)]
        image = image / 10000.0 if image.quantile(.99) > 2 else image
        image = torch.nan_to_num(image).clamp(0,1)
        invalid = ~torch.isfinite(sample['image']).all(0); mask[invalid] = 255
        if self.augment:
            if torch.rand(()) < .5: image, mask = image.flip(-1), mask.flip(-1)
            if torch.rand(()) < .5: image, mask = image.flip(-2), mask.flip(-2)
            k=int(torch.randint(0,4,())); image,mask=torch.rot90(image,k,(-2,-1)),torch.rot90(mask,k,(-2,-1))
            image = (image * torch.empty(1).uniform_(.9,1.1) + torch.empty(1).uniform_(-.03,.03)).clamp(0,1)
        return image, mask, geography_key(name)
loaders = {s:DataLoader(GroupedDataset(rows,s=='train'), batch_size=CFG['batch_size'], shuffle=s=='train', num_workers=2, pin_memory=True) for s,rows in partitions.items()}

In [ ]:
# SatlasPretrain multispectral backbone+FPN; only supported 9 bands are routed, in documented order.
import satlaspretrain_models, torch.nn as nn
S2_ALL = ['B01','B02','B03','B04','B05','B06','B07','B08','B8A','B09','B10','B11','B12']
SATLAS = ['B04','B03','B02','B05','B06','B07','B08','B11','B12']
SATLAS_IDX = [S2_ALL.index(x) for x in SATLAS]
class LandCoverNet(nn.Module):
    def __init__(self):
        super().__init__(); self.encoder = satlaspretrain_models.Weights().get_pretrained_model('Sentinel2_SwinB_SI_MS', fpn=True)
        self.head = nn.Sequential(nn.Conv2d(128*4,256,3,padding=1),nn.BatchNorm2d(256),nn.ReLU(),nn.Dropout2d(.1),nn.Conv2d(256,CFG['num_classes'],1))
    def forward(self,x):
        size=x.shape[-2:]; features=self.encoder(x[:,SATLAS_IDX]); base=features[0].shape[-2:]
        merged=torch.cat([features[0], *[F.interpolate(f,base,mode='bilinear',align_corners=False) for f in features[1:]]],1)
        return F.interpolate(self.head(merged),size,mode='bilinear',align_corners=False)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'); model=LandCoverNet().to(device)
print('device',device,'parameters',sum(p.numel() for p in model.parameters()))

In [ ]:
# Weighted cross-entropy/focal + Dice, AMP, checkpointing, automatic resume, and early stopping.
from collections import defaultdict
weights=torch.tensor([.2,1.5,.8,1.3,1.4,1.2,2.0],device=device)
def loss_fn(logits,target):
    ce=F.cross_entropy(logits,target,weight=weights,ignore_index=255,reduction='none'); valid=target!=255
    ce=(((1-torch.exp(-ce))**2)*ce)[valid].mean()
    safe=target.masked_fill(~valid,0); one=F.one_hot(safe,7).permute(0,3,1,2).float()*valid[:,None]
    p=logits.softmax(1); dice=1-((2*(p*one).sum((0,2,3))+1e-6)/((p*valid[:,None]).sum((0,2,3))+one.sum((0,2,3))+1e-6)).mean()
    return .5*ce+.5*dice
optimizer=torch.optim.AdamW(model.parameters(),lr=CFG['lr'],weight_decay=CFG['weight_decay']); scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,CFG['epochs'])
scaler=torch.amp.GradScaler('cuda',enabled=device.type=='cuda'); ckpt=ROOT/'last_checkpoint.pt'; best_path=ROOT/'best_checkpoint.pt'; start,best,stale=0,-1,0
if ckpt.exists():
    state=torch.load(ckpt,map_location='cpu',weights_only=False); model.load_state_dict(state['model']); optimizer.load_state_dict(state['optimizer']); scheduler.load_state_dict(state['scheduler']); scaler.load_state_dict(state['scaler']); start,best,stale=state['epoch']+1,state['best'],state['stale']
def score(loader):
    model.eval(); inter=torch.zeros(7,device=device); union=torch.zeros(7,device=device)
    with torch.no_grad():
      for image,target,_ in loader:
        pred=model(image.to(device)).argmax(1); target=target.to(device)
        for c in range(7): inter[c]+=((pred==c)&(target==c)).sum(); union[c]+=((pred==c)|(target==c)).sum()
    supported=union>0; return ((inter[supported]+1e-6)/(union[supported]+1e-6)).mean().item()
for epoch in range(start,CFG['epochs']):
    model.train()
    for image,target,_ in loaders['train']:
      optimizer.zero_grad(set_to_none=True); image,target=image.to(device),target.to(device)
      with torch.autocast(device_type=device.type,dtype=torch.float16,enabled=device.type=='cuda'): loss=loss_fn(model(image),target)
      scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
    scheduler.step(); miou=score(loaders['val']); stale=0 if miou>best else stale+1
    state={'model':model.state_dict(),'optimizer':optimizer.state_dict(),'scheduler':scheduler.state_dict(),'scaler':scaler.state_dict(),'epoch':epoch,'best':max(best,miou),'stale':stale,'config':CFG}
    torch.save(state,ckpt)
    if miou>best: best=miou; torch.save(state,best_path)
    print(epoch,{'loss':float(loss),'val_mIoU':miou,'best':best,'stale':stale})
    if stale>=CFG['patience']: break

In [ ]:
# Held-out, per-geography evaluation and calibration. No numbers are written unless computed here.
state=torch.load(best_path,map_location=device,weights_only=False); model.load_state_dict(state['model']); model.eval()
# Tune abstention confidence only on validation. Coverage-weighted mIoU prevents a high threshold from winning by discarding difficult pixels.
val_prob,val_target=[],[]
with torch.no_grad():
  for image,target,_ in loaders['val']: val_prob.append(model(image.to(device)).softmax(1).cpu()); val_target.append(target)
val_prob,val_target=torch.cat(val_prob),torch.cat(val_target); val_pred=val_prob.argmax(1); val_conf=val_prob.max(1).values; candidates=[]
for threshold in torch.linspace(0,.9,19):
  accepted=(val_conf>=threshold)&(val_target!=255); ious=[]
  for c in range(7):
    union=((val_pred==c)|(val_target==c))&accepted
    if union.any(): ious.append((((val_pred==c)&(val_target==c)&accepted).sum()/union.sum()).item())
  coverage=accepted.sum().item()/max(1,(val_target!=255).sum().item()); miou=float(np.mean(ious)) if ious else 0.; candidates.append({'threshold':float(threshold),'macro_iou':miou,'coverage':coverage,'objective':miou*coverage})
calibration=max(candidates,key=lambda x:x['objective'])
scene=defaultdict(lambda:{'intersection':torch.zeros(7),'union':torch.zeros(7),'pixels':0}); confidence,target_all=[],[]
with torch.no_grad():
  for image,target,geographies in loaders['test']:
    probs=model(image.to(device)).softmax(1).cpu(); pred=probs.argmax(1)
    confidence.append(probs.max(1).values); target_all.append(target)
    for i,g in enumerate(geographies):
      for c in range(7): scene[g]['intersection'][c]+=((pred[i]==c)&(target[i]==c)).sum(); scene[g]['union'][c]+=((pred[i]==c)|(target[i]==c)).sum()
      scene[g]['pixels']+=int((target[i]!=255).sum())
scene_metrics={g:{'per_class_iou':((v['intersection']+1e-6)/(v['union']+1e-6)).tolist(),'macro_iou':float(((v['intersection'][v['union']>0]+1e-6)/(v['union'][v['union']>0]+1e-6)).mean()),'pixels':v['pixels']} for g,v in scene.items()}
# Calibration candidates are evaluated on validation in a production run; test remains untouched.
metrics={'mode':MODE,'best_validation_miou':best,'validation_calibration':calibration,'test_by_geography':scene_metrics,'unsupported_class_warning':'DFC2020 lacks a separate road mask; add geographically held-out road labels before production registration.'}
(ROOT/'metrics.json').write_text(__import__('json').dumps(metrics,indent=2)); metrics

In [ ]:
# Export model, exact config, class map, metrics, model card, inference contract, and a downloadable ZIP.
import json, zipfile
class_map={'0':'unknown','1':'built-up','2':'vegetation','3':'cropland','4':'bare soil','5':'water','6':'road or impervious surface','ignore_index':255}
(ROOT/'class_mapping.json').write_text(json.dumps(class_map,indent=2)); (ROOT/'training_config.json').write_text(json.dumps(CFG,indent=2))
card=f'''# SatQuery Sentinel-2 land cover
Base: SatlasPretrain Sentinel2_SwinB_SI_MS. Dataset: DFC2020-S2 with SHA/MD5-verified download and geographic grouping. Run mode: {MODE}. Best validation mIoU: {best:.6f}. Test details: metrics.json. Limitation: DFC2020 has no independent road mask; this checkpoint MUST NOT be registered for production until that class receives representative labels and held-out evaluation.'''
(ROOT/'MODEL_CARD.md').write_text(card)
example='python -m satquery.cli INPUT_13B.tif OUTPUT --band-order B01 B02 B03 B04 B05 B06 B07 B08 B8A B09 B10 B11 B12'
(ROOT/'inference_example.txt').write_text(example)
archive=ROOT/'satquery_landcover_sentinel2_export.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
  for name in ('best_checkpoint.pt','training_config.json','metrics.json','class_mapping.json','MODEL_CARD.md','inference_example.txt'): z.write(ROOT/name,arcname=name)
print(archive)
try:
 from google.colab import files; files.download(str(archive))
except ImportError: pass